# Implementing Self Attention Mechanism using trainable weights

In [1]:
import torch

In [2]:
inputs=torch.tensor(
    [[0.43, 0.15, 0.89], # Your    (x^1)
     [0.55, 0.87, 0.66], # journey (x^2)
     [0.57, 0.85, 0.64], # starts  (x^3)
     [0.22, 0.58, 0.33], # with    (x^4)
     [0.77, 0.25, 0.10], # one     (x^5)
     [0.05, 0.80, 0.55]] # step    (x^6)
)

In [3]:
x_2=inputs[1]
d_in=inputs.shape[1]
d_out=2

Usually the input and output dimensions are same in models like the GPT models but here we are using the (d_in=3) and (d_out=2) for illustrations purposes to better follow the computations

In [4]:
torch.manual_seed(123)
W_query=torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key=torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value=torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [5]:
print(W_query)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])


In [6]:
print(W_key)

Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])


In [7]:
print(W_value)

Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])


## Calculating the query, key and value matrices for the query 2 which is journey

In [8]:
query_2=x_2 @ W_query
key_2=x_2 @ W_key
value_2=x_2 @ W_value

In [9]:
print(query_2)

tensor([0.4306, 1.4551])


In [10]:
print(key_2)

tensor([0.4433, 1.1419])


In [11]:
print(value_2)

tensor([0.3951, 1.0037])


## Now we will obtain the Queries, Keys and Values matrices for all the input embedding vectors

In [12]:
queries=inputs @ W_query
keys=inputs @ W_key
values=inputs @ W_value

print("queries.shape:", queries.shape)
print("keys.shape: ", keys.shape)
print("values.shape:", values.shape)

queries.shape: torch.Size([6, 2])
keys.shape:  torch.Size([6, 2])
values.shape: torch.Size([6, 2])


As we can see the shape here, we have successfully projected the 6 input tokens from 3D space to a 2D embedding space

## First let us calculate the attention score w22

In [13]:
key_2=keys[1]
attention_score_22=query_2.dot(key_2)
print(attention_score_22)

tensor(1.8524)


## Now let us just compute the attention scores for the 2nd query from the query matric which is "journey"

In [14]:
attention_scores_2=query_2 @ keys.T # All attention scores for the given query="journey"
print(attention_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [15]:
attention_scores=queries @ keys.T
print(attention_scores)

tensor([[0.9231, 1.3545, 1.3241, 0.7910, 0.4032, 1.1330],
        [1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
        [1.2544, 1.8284, 1.7877, 1.0654, 0.5508, 1.5238],
        [0.6973, 1.0167, 0.9941, 0.5925, 0.3061, 0.8475],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707, 0.7307],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]])


## Before normalization with softmax we scale it by sqrt(dimensions of the keys matrix) which is 2 in our case and then we apply the softmax function

In [16]:
d_keys=keys.shape[-1]
attention_weights_2=torch.softmax(attention_scores_2/d_keys**0.5, dim=-1)
print(attention_weights_2)
print(f"Dimentions of keys matrics: {d_keys}")

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
Dimentions of keys matrics: 2


### But why do we scale it by sqrt(dimensions of the keys matrix)?
- Reason 1: For stability in learning
- Reason 2: To make the variance of the dot product stable

In [17]:
# Demonstration for reason 1
tensor=torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])

# Apply softmax without scaling
softmax_result=torch.softmax(tensor, dim=-1)
print(f"Softmax without scaling: {softmax_result}")

# Multiply the tensor by 8 and then apply the softmax
scaled_tensor=tensor*8
softmax_scaled_result=torch.softmax(scaled_tensor, dim=-1)
print(f"Softmax after scaling (tensor*8): {softmax_scaled_result}")

Softmax without scaling: tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
Softmax after scaling (tensor*8): tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


In [18]:
# Demonstration for reason 2
import numpy as np

def compute_variance(dim, num_trials=1000):
    dot_products=[]
    scaled_dot_products=[]

    # Generate multiple random vectors and compute dot products
    for _ in range(num_trials):
        Q=np.random.randn(dim)
        K=np.random.randn(dim)

        # Compute the dot product
        dot_product=np.dot(Q, K)
        dot_products.append(dot_product)

        # Scale the dot product by sqrt(dim)
        scaled_dot_product=dot_product/np.sqrt(dim)
        scaled_dot_products.append(scaled_dot_product)

    variance_before_scaling=np.var(dot_products)
    variance_after_scaling=np.var(scaled_dot_products)

    return variance_before_scaling, variance_after_scaling

# For dimension 5
v_before_5, v_after_5=compute_variance(5)
print(f"Vairance before scaling (dim=5): {v_before_5}")
print(f"Vairance after scaling (dim=5): {v_after_5}")

# For dimension 20
v_before_20, v_after_20=compute_variance(20)
print(f"Vairance before scaling (dim=20): {v_before_20}")
print(f"Vairance after scaling (dim=20): {v_after_20}")

# For dimension 20
v_before_100, v_after_100=compute_variance(100)
print(f"Vairance before scaling (dim=100): {v_before_100}")
print(f"Vairance after scaling (dim=100): {v_after_100}")

Vairance before scaling (dim=5): 4.671392634857029
Vairance after scaling (dim=5): 0.9342785269714061
Vairance before scaling (dim=20): 20.492222302142125
Vairance after scaling (dim=20): 1.0246111151071058
Vairance before scaling (dim=100): 90.73375157976113
Vairance after scaling (dim=100): 0.9073375157976113


## Computing the context vector for the query(2) which is journey

In [19]:
context_vector_2=attention_weights_2 @ values
print(context_vector_2)

tensor([0.3061, 0.8210])


## Implementing the SelfAttention_V1 class to create the context vectors for all the queries

In [21]:
import torch
from torch import nn
import numpy as np

In [22]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query=nn.Parameter(torch.rand(d_in, d_out))
        self.W_key=nn.Parameter(torch.rand(d_in, d_out))
        self.W_value=nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        queries=x @ self.W_query
        keys=x @ self.W_key
        values=x @ self.W_value

        attention_scores=queries @ keys.T
        attention_weights=torch.softmax(attention_scores/np.sqrt(keys.shape[-1]), dim=-1)

        context_vectors=attention_weights @ values
        return context_vectors

In [23]:
torch.manual_seed(123)
self_attention_v1=SelfAttention_v1(d_in, d_out)
print(self_attention_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In this PyTorch code, SelfAttention_v1 is a class derived from nn.Module, which is a
fundamental building block of PyTorch models, which provides necessary functionalities for
model layer creation and management.

The __init__ method initializes trainable weight matrices (W_query, W_key, and
W_value) for queries, keys, and values, each transforming the input dimension d_in to an
output dimension d_out.

During the forward pass, using the forward method, we compute the attention scores
(attn_scores) by multiplying queries and keys, normalizing these scores using softmax.

## SelfAttention_v2 using the nn.Linear instead of nn.Parameter

We can improve the SelfAttention_v1 implementation further by utilizing PyTorch's
nn.Linear layers, which effectively perform matrix multiplication when the bias units are
disabled. 

Additionally, a significant advantage of using nn.Linear instead of manually
implementing nn.Parameter(torch.rand(...)) is that nn.Linear has an optimized weight
initialization scheme, contributing to more stable and effective model training.

In [26]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value=nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries=self.W_query(x)
        keys=self.W_key(x)
        values=self.W_value(x)

        attention_scores=queries @ keys.T
        attention_weights=torch.softmax(attention_scores/np.sqrt(keys.shape[-1]), dim=-1)

        context_vectors=attention_weights @ values
        return context_vectors

In [27]:
torch.manual_seed(789)
self_attention_v2=SelfAttention_v2(d_in, d_out)
print(self_attention_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
